<a href="https://colab.research.google.com/github/jonik2909/jaydariGPT/blob/main/jaydari_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch bitsandbytes datasets peft trl

In [ ]:
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, TrainingArguments
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

In [ ]:
# 1. Configuration and Tokenizer
model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
tokenizer = AutoTokenizer.from_pretrained(model_id)

# print('Vocab size:', tokenizer.vocab_size)
# print('Special tokens:', tokenizer.special_tokens_map)

# 2. Quantization Setup (4-bit)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4'
)

# 3. Load Model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto', # Automatically handles GPU/CPU placement
    # dtype=torch.bfloat16
)

In [ ]:
# 4. Inference Test (Before Fine-tuning)
# prompt = "Explain what a tokenizer is."
prompt = "A tokenizer is a tool in natural language processing that"

# Prepare inputs and move to the same device as the model
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate output
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7
    )

# Decode and print results
print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

# first_block = model.model.layers[0]
# print("first_block:", first_block)
# print(first_block.self_attn)
# print(model.config)

In [ ]:
def count_parameters(model):
  return sum(p.numel() for p in model.parameters())

total_params = count_parameters(model)
print(f"Total parameters (including fronzen 4-bit): {total_params:,}")

## datasets library | load_dataset
* intruction tuning

In [ ]:
dataset = load_dataset("yahma/alpaca-cleaned", split="train")
dataset[0]

In [ ]:
def generate_prompt(example):
  intsruction = example['instruction']
  input_text = example['input']
  output_Text = example['output']

  if input_text:
    return (
        "### Instruction:\n"
        f"{intsruction}\n\n"
        "### Input:\n"
        f"{input_text}\n\n"
        "### Response:\n"
        f"{output_Text}\n\n"
    )
  else:
    return(
        "### Instruction:\n"
        f"{intsruction}\n\n"
        "### Response:\n"
        f"{output_Text}\n\n"
    )

# generate_prompt(dataset[0])

def formatting_func(example):
  return {'text': generate_prompt(example)}

dataset = dataset.map(formatting_func)

In [ ]:
dataset[0]['text']

In [ ]:
dataset = dataset.select(range(7000))

In [ ]:
dataset = dataset.shuffle(seed=42)

In [ ]:
dataset

In [ ]:
# Full Fine-tuning =>
# Cheap Fine-tuning =>
# PEFT => Parameter Efficient Fine Tuning
# OOM => Out of Memory

In [ ]:
# Define the LoRA Configuration
lora_config = LoraConfig(
    r=8,                # Rank: lower numbers mean fewer parameters
    lora_alpha=16,      # Scaling factor for the learned weights
    lora_dropout=0.05,  # Dropout probability to prevent overfitting
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"], # Targets the Attention Query and Value matrices
)

In [ ]:
# Wrap the base model with LoRA layers
model = get_peft_model(model, lora_config)

In [ ]:
# Verify the reduction in trainable parameters
model.print_trainable_parameters()

In [ ]:
# 1. Define Training Arguments
training_args = TrainingArguments(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4, # VRAM optimization
    num_train_epochs=2,            # overfit
    logging_steps=20,
    output_dir="./jaydari_gpt",
    save_strategy="epoch",
    bf16=True,
    fp16=False,
    report_to="none"
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    formatting_func=lambda x: x['text'],
    args=training_args
)

# 4. Start Training and Save
trainer.train()
model.save_pretrained('jaydari_gpt')
tokenizer.save_pretrained('jaydari_gpt')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_dir = "/content/jaydari_gpt"

# Load tokenizer and configure padding
tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
tokenizer.pad_token = tokenizer.eos_token

# Load the base model with bfloat16 precision for efficiency
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

# Load the fine-tuned adapter and set the model to evaluation mode
model = PeftModel.from_pretrained(base_model, adapter_dir)
model.eval()

# **TESTING BASE MODEL**

In [ ]:
prompt = "Explain what machine learning is in simple words."

# Tokenize the prompt and move it to the same device as the base model
inputs_base = tokenizer_base(prompt, return_tensors="pt").to(model_base.device)

# Generate response without calculating gradients (to save memory)
with torch.no_grad():
    output_base = model_base.generate(
        **inputs_base,
        max_new_tokens=120,
        temperature=0.7,
        do_sample=True
    )

# Decode and print the base model's output
print("===== BASE MODEL OUTPUT =====")
print(tokenizer_base.decode(output_base[0], skip_special_tokens=True))

# **FINE-TUNED MODEL INFRANCE**

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "jaydari_gpt"

# Load the saved fine-tuned tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_path)

model_ft = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto"
)

model_ft.eval()

In [ ]:
import torch

# Define a prompt using the specific instruction-response format
prompt = """### Instruction:
Explain what machine learning is in simple words.

### Response:
"""

# Tokenize the formatted prompt
inputs_ft = tokenizer_ft(prompt, return_tensors="pt").to(model_ft.device)

# Generate response using the fine-tuned model
with torch.no_grad():
    output_ft = model_ft.generate(
        **inputs_ft,
        max_new_tokens=120,
        temperature=0.7,
        do_sample=True
    )

# Display the final results
print("===== FINE-TUNED MODEL OUTPUT =====")
print(tokenizer_ft.decode(output_ft[0], skip_special_tokens=True))

# **ZIP AND DOWNLOAD**

In [ ]:
!zip -r jaydari_gpt.zip jaydari_gpt

In [ ]:
from google.colab import files
files.download("jaydari_gpt.zip")

In [ ]:
######################
#     MODEL TEST     #
######################

In [ ]:
# Install the language detection library
!pip install -q langdetect

In [ ]:
# Unzip your saved model adapter
!unzip jaydari_gpt.zip

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_dir = "/content/jaydari_gpt"

# Load tokenizer and configure padding
tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
tokenizer.pad_token = tokenizer.eos_token

# Load the base model with bfloat16 precision for efficiency
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

# Load the fine-tuned adapter and set the model to evaluation mode
model = PeftModel.from_pretrained(base_model, adapter_dir)
model.eval()

In [ ]:
from langdetect import detect_langs

def ask_jaydari_gpt(prompt: str, max_new_tokens=256):
    try:
        # Detect the input language
        langs = detect_langs(prompt)
        lang_code = langs[0].lang
        confidence = langs[0].prob
        print(f"🌍 Detected lang: {lang_code} ({confidence:.2f})")
    except:
        lang_code = "unknown"
        confidence = 0.0

    # Block non-English prompts if the detection confidence is high
    if lang_code != "en" and confidence >= 0.95 and len(prompt) > 20:
        return "Sorry, you should ask a question only in English."

    # Wrap the user prompt in the training instruction format
    full_prompt = f"### Instruction:\n{prompt.strip()}\n\n### Response:"
    inputs = tokenizer(full_prompt, return_tensors="pt").to("cuda")

    # Generate response with sampling
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id
    )

    # Return only the newly generated text
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded[len(full_prompt):].strip()

In [ ]:
ask_jaydari_gpt("What is Javascript?")